# MB4 · Rang faible et recherche d'information — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/mb4_rang_faible/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). Dans les parties A à D, donne les valeurs que rend le calcul, **sans les arrondir** : les décimales demandées sont celles de ta copie. « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/mb4_rang_faible/03_notebook.ipynb`, créée par `python tools/start_chapter.py MB4`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | MB4.1 à MB4.4 | Les exercices papier (vérification) | ✏️ | | |
| A | MB4.8 | La factorisation de rang C F : les colonnes pivots de M et les lignes non nulles de sa forme échelonnée réduite · `rank_factorization` | 🔨 | ★ | 20 |
|  | MB4.9 | Les points de Borda de classements complets, et le MRR exact d'un ordre tiré au hasard, en fractions · `borda_scores`, `random_ranking_mrr` | 🔨 | ★★ | 25 |
| B | MB4.10 | Prédire le MRR d'un ordre tiré au hasard parmi 10, 100 puis 1 000 candidats, puis le simuler et le comparer à H_n/n | 🔮 | ★ | 15 |
|  | MB4.11 | Les courbes de saturation x ↦ x(k₁ + 1)/(x + k₁) de k₁ = 0 à l'infini, et ce que la normalisation par la longueur leur fait | 🎨 | ★ | 15 |
| C | MB4.12 | Chasse à l'erreur : C pris dans la forme réduite au lieu de M, un classement lu comme la liste des rangs des candidats, un poids de rareté écrit ln(n/N) | 🐛 | ★ | 15 |
| D | MB4.13 | Défi : retrouver le rang caché de cinq tableaux bruités, cinq fois sur cinq, par leurs valeurs singulières | 🏆 | ★★★ | 25 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB4")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : les ventes de quatre boulangeries, les relevés de stations météo, les ingrédients d'un carnet de recettes, et trois jurys d'un concours de pâtisserie.
- Écrire les trois fonctions de `mymath/ranks.py` (`rank_factorization`, `borda_scores`, `random_ranking_mrr`) et les tester.
- Prédire puis simuler le MRR d'un ordre tiré au hasard, tracer des courbes de saturation, corriger trois erreurs classiques, et relever un défi.

**Rappel express.** Une matrice $(m, n)$ de rang $r$ est un produit $\mathbf{C}\mathbf{F}$ : $\mathbf{C}$ ses $r$ colonnes pivots, $\mathbf{F}$ les $r$ lignes non nulles de sa forme échelonnée réduite ; $r(m + n)$ nombres stockés, $r(m + n - r)$ libres ; $\mathbf{C}(\mathbf{F}\mathbf{x})$ coûte $r(m + n)$ multiplications. L'énergie $\lVert\mathbf{M}\rVert_F^2 = \sum_is_i^2$ ; le rang effectif à $\tau$, le plus petit $r$ dont la part atteint $\tau$ ; le rang stable $\sum_is_i^2/s_1^2$. La surprise $-\ln\frac nN$, le poids corrigé $\ln\frac{N - n + 1/2}{n + 1/2}$. La saturation $s(x) = \frac{x(k_1 + 1)}{x + k_1}$, et la normalisation $\nu = 1 - b + b\ell$, qui remplace $k_1$ par $k_1\nu$. Borda : $n - j$ points au rang $j$ ; la somme des $\frac{1}{c + \text{rang}}$. Pour un ordre tiré au hasard, $P(R = j) = \binom{n - j}{q - 1}\big/\binom nq$.

## Partie 0 · Vérifier tes exercices papier (✏️ MB4.1 à MB4.4)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un entier, ou un nombre avec un point décimal, `-1.25`, arrondi comme le demande la question (« a number » sans décimales : la valeur exacte, ce calcul tombe juste) ;
- une fraction irréductible entre guillemets, `"5/8"` ;
- une liste `[0.2, -1]`, dans l'ordre que dit la question ; une matrice comme la liste de ses **lignes**, `[[1, 0], [-2, 3]]` ; une matrice de valeurs exactes peut contenir des fractions entre guillemets.

Les réponses pas encore remplies affichent ⏳.

**Ex MB4.1 — Les ventes de quatre boulangeries sur cinq produits : le rang, la factorisation par les colonnes pivots, une seconde factorisation (CG)(G⁻¹F), et les nombres stockés ou libres**

In [ ]:
# Ex MB4.1: the values you found on paper
answer_MB4_1a = ...  # a) an integer
answer_MB4_1b = ...  # b) a 2 × 5 matrix, as the list of its rows (exact values)
answer_MB4_1c = ...  # c) a 4 × 2 matrix, as the list of its rows
answer_MB4_1d = ...  # d) a 2 × 5 matrix, as the list of its rows (exact values)
answer_MB4_1e = ...  # e) an integer
answer_MB4_1f = ...  # f) an integer
answer_MB4_1g = ...  # g) a list of 2 integers: the numbers stored, then the free numbers

for letter, answer in zip("abcdefg", [answer_MB4_1a, answer_MB4_1b, answer_MB4_1c, answer_MB4_1d, answer_MB4_1e, answer_MB4_1f, answer_MB4_1g]):
    wbm.check(f"MB4.1{letter}", answer)

**Ex MB4.2 — Un tableau de relevés de stations météo donné par ses valeurs singulières : énergie captée, rang effectif à 99 %, rang stable, puis le coût de C(Fx) et de (CF)x pour un et pour mille vecteurs**

In [ ]:
# Ex MB4.2: the values you found on paper
answer_MB4_2a = ...  # a) an integer
answer_MB4_2b = ...  # b) a number (4 decimals)
answer_MB4_2c = ...  # c) an integer
answer_MB4_2d = ...  # d) a number (2 decimals)
answer_MB4_2e = ...  # e) a number (3 decimals)
answer_MB4_2f = ...  # f) an integer
answer_MB4_2g = ...  # g) an integer
answer_MB4_2h = ...  # h) a list of 2 integers: C(Fx), then (CF)x

for letter, answer in zip("abcdefgh", [answer_MB4_2a, answer_MB4_2b, answer_MB4_2c, answer_MB4_2d, answer_MB4_2e, answer_MB4_2f, answer_MB4_2g, answer_MB4_2h]):
    wbm.check(f"MB4.2{letter}", answer)

**Ex MB4.3 — Les ingrédients d'un carnet de recettes : surprise, cote, poids corrigé par ½ et son signe, la variante toujours positive, puis la saturation et la normalisation d'une mention répétée**

In [ ]:
# Ex MB4.3: the values you found on paper
answer_MB4_3a = ...  # a) a number (4 decimals)
answer_MB4_3b = ...  # b) an irreducible fraction, between quotes
answer_MB4_3c = ...  # c) a number (4 decimals)
answer_MB4_3d = ...  # d) a number (4 decimals)
answer_MB4_3e = ...  # e) an integer
answer_MB4_3f = ...  # f) a number (4 decimals)
answer_MB4_3g = ...  # g) a list of 3 numbers (4 decimals): s(1), s(2), s(10)
answer_MB4_3h = ...  # h) an irreducible fraction, between quotes
answer_MB4_3i = ...  # i) a list of 2 numbers (4 decimals): b = ¼, then b = ½

for letter, answer in zip("abcdefghi", [answer_MB4_3a, answer_MB4_3b, answer_MB4_3c, answer_MB4_3d, answer_MB4_3e, answer_MB4_3f, answer_MB4_3g, answer_MB4_3h, answer_MB4_3i]):
    wbm.check(f"MB4.3{letter}", answer)

**Ex MB4.4 — Trois jurys classent cinq pâtissiers : les points de Borda, la somme des 1/(c + rang) pour c = 2 et c = 10, et la somme des notes qui change de gagnant quand un jury passe de 20 à 100 points**

In [ ]:
# Ex MB4.4: the values you found on paper
answer_MB4_4a = ...  # a) a list of 5 integers: A, B, C, D, E
answer_MB4_4b = ...  # b) a list of 5 numbers (4 decimals): A to E, for c = 2
answer_MB4_4c = ...  # c) a list of 5 numbers (4 decimals): A to E, for c = 10
answer_MB4_4d = ...  # d) a list of 5 integers: A to E
answer_MB4_4e = ...  # e) a list of 5 integers: A to E
answer_MB4_4f = ...  # f) a number

for letter, answer in zip("abcdef", [answer_MB4_4a, answer_MB4_4b, answer_MB4_4c, answer_MB4_4d, answer_MB4_4e, answer_MB4_4f]):
    wbm.check(f"MB4.4{letter}", answer)

## Partie A · Ta librairie : la factorisation de rang, les points de Borda et le MRR d'un ordre au hasard

Fiche §1, §5 et §6. Les trois fonctions de `mymath/ranks.py`, leurs tests, et un usage de chacune. La cellule de données ci-dessous recopie les données des exercices papier.

In [ ]:
# Tools of the notebook (parts A to D)
import math
import re
import time

import matplotlib.pyplot as plt
import numpy as np

from wbm.errors import MymathMissingError
from wbm.nbtools import StopCheck, checked, drawn, filled, fr, run_pytest_group, verdict

TEST_MB4 = "tests/test_mb4_ranks.py"               # the tests of mymath/ranks.py
REF = wbm.load_mymath("ref", alias="mymath_ref")  # the reference: the checks of the hidden cases rely on it
ref_ranks = REF.ranks


def shown(value, digits=4):
    """A short text for a result of the learner's code (the check messages): numbers rounded, as lists."""
    if value is None:
        return "None"
    try:
        return str(np.round(np.asarray(value, dtype=float), digits).tolist())[:160]
    except (TypeError, ValueError):
        return repr(value)[:160]


def check_answer(ex_id, value, hint=None):
    """wbm.check(ex_id, value, computed=True). For a list that holds True or False, the message gives only how many
    elements are right, not the place of the first wrong one (it would let you guess the booleans one by one).
    hint(value), when given, may return a targeted « Piste » for a wrong answer (None otherwise)."""
    holds_bool = isinstance(value, (list, tuple)) and any(isinstance(v, (bool, np.bool_)) for v in value)
    if not holds_bool and hint is None:
        return wbm.check(ex_id, value, computed=True)
    result = wbm.check(ex_id, value, computed=True, quiet=True)
    message = result.message
    if holds_bool:
        for cut in (" Le premier faux est le ", " Premier élément faux à l'indice "):
            message = message.split(cut)[0]
    if hint is not None and result.status not in ("correct", "pending", "unknown") and "Piste" not in message:
        try:
            extra = hint(value)
        except Exception:  # noqa: BLE001 - an odd answer: no targeted hint
            extra = None
        if extra:
            message = f"{message} Piste : {extra}"
    icon = {"correct": "✅", "pending": "⏳", "unknown": "❓"}.get(result.status, "❌")
    print(f"{icon} Ex {ex_id} : {message}")
    return result


def passes(ex_id, value):
    """True when value is the hidden answer ex_id (wbm.check, computed=True, nothing printed)."""
    try:
        return bool(wbm.check(ex_id, value, computed=True, quiet=True).passed)
    except Exception:  # noqa: BLE001 - an odd value is not the answer
        return False


def reporter(results, stop=True):
    """A function report(ok, success, failure) that prints one ✅ or ❌ line, keeps the verdict in results, and (stop)
    ends the check at the first ❌ (StopCheck, caught by checked)."""
    def report(ok, success, failure):
        results.append(bool(ok))
        print(("✅ " + success) if ok else ("❌ " + failure))
        if not ok and stop:
            raise StopCheck
    return report


def as_floats(value):
    """value as a float array, or None when it is not an array of numbers."""
    try:
        return np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        return None


def matches(got, want, rtol=1e-9, atol=1e-12):
    """True when got is an array of numbers of the shape of want and equal to it up to rounding (no error for an odd
    got)."""
    arr = as_floats(got)
    want = np.asarray(want, dtype=float)
    return arr is not None and arr.shape == want.shape and bool(np.allclose(arr, want, rtol=rtol, atol=atol,
                                                                            equal_nan=True))


def line_with(ax, xs, ys, rtol=1e-7, atol=1e-9):
    """True when a line of ax has exactly the abscissas xs and the ordinates ys."""
    xs, ys = np.asarray(xs, dtype=float), np.asarray(ys, dtype=float)
    for line in ax.lines:
        x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
        if x.shape == xs.shape and y.shape == ys.shape and np.allclose(x, xs, rtol=rtol, atol=atol) \
                and np.allclose(y, ys, rtol=rtol, atol=atol, equal_nan=True):
            return True
    return False


def call_safely(label, function, *args, **kwargs):
    """function(*args, **kwargs); an error of the learner's code becomes one ❌ line naming the call (StopCheck)."""
    try:
        return function(*args, **kwargs)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the call
        print(f"❌ {label} : ton code lève {type(exc).__name__} : {str(exc)[:200]}")
        raise StopCheck from None


# The data of the paper exercises (02_exercices.md)
M_1 = np.array([[20.0, 6, 16, 12, 8], [8, 10, 14, 20, 7], [12, 4, 10, 8, 5], [16, 2, 10, 4, 5]])   # MB4.1: the sales (in
#   tens) of the bakeries Port, Gare, Halles, Moulin (rows) for baguettes, croissants, pains de campagne, pains au
#   chocolat, tartelettes (columns)
S_2 = np.array([320.0, 150.0, 60.0, 15.0] + [3.0] * 56)     # MB4.2: the singular values of the 60 x 450 table
RANKINGS_4 = np.array([[0, 1, 2, 3, 4], [0, 1, 3, 4, 2], [2, 1, 4, 3, 0]])   # MB4.4: the rankings of the juries of the
#   chefs, of the public and of the press, best first (the pastry cooks A, B, C, D, E numbered 0 to 4)
for _array in (M_1, S_2, RANKINGS_4):
    _array.setflags(write=False)   # read only: a change in place raises an error at the line that does it

### Ex MB4.8 — La factorisation de rang C F : les colonnes pivots de M et les lignes non nulles de sa forme échelonnée réduite · `rank_factorization` 🔨 ★ ⏱️ 20 min
**Objectif :** programmer la factorisation de rang par les colonnes pivots, puis retrouver MB4.1 et voir le rôle de la tolérance sur un tableau mesuré.  
**Prérequis :** fiche §1 ; M09a (`rref`), M12a (colonnes pivots, rang numérique) ; Ex MB4.1 · **mymath :** `ranks.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme en MB3). Ouvre ta copie `mon_travail/mymath/ranks.py`, lis la docstring du module (ses conventions, et ce qui est permis) puis celle de la fonction, remplace chaque `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Dans le notebook, appelle tes fonctions par `mymath.ranks.<nom>` (par exemple `mymath.ranks.rank_factorization(M)`), ou donne-leur un nom court (`rank_factorization = mymath.ranks.rank_factorization`) dans une cellule exécutée après le rechargement. ⚠️ Le squelette n'importe rien : ajoute en tête du fichier `import math`, `from fractions import Fraction` et `import numpy as np`, et `from .matrices import rref` si tu reprends ton `rref` de M09a (sinon, recopie-le dans une fonction privée `_rref`). NumPy est permis (`@` compris), mais rien de `np.linalg`, ni SciPy (`scipy.linalg`, `scipy.stats.rankdata`), ni SymPy : les tests comparent tes résultats aux leurs, et un test lit ton module ; dans `borda_scores`, ni `np.argsort` ni `np.lexsort`. Les candidats sont numérotés à partir de 0, un rang commence à 1 ; ne modifie jamais les tableaux reçus. Hors de `mymath`, le notebook peut se servir de tout NumPy et de SciPy.

Écris `rank_factorization(M, tol=1e-10)` (fiche §1), en suivant sa docstring : contrôle $\mathbf{M}$ (un tableau 2-D non vide de nombres finis) et `tol` (un réel $\ge 0$, ni `nan` ni un booléen) ; réduis une copie en flottants comme le fait `rref` de M09a (tu peux importer `rref` de `.matrices`, ou le recopier dans une fonction privée `_rref`), et garde ses colonnes pivots ; $\mathbf{C}$ est faite des colonnes pivots de $\mathbf{M}$ elle-même, $\mathbf{F}$ des $r$ premières lignes de la forme réduite.

**L'usage.** Le notebook fournit `M_8`, les ventes moyennes d'une semaine de 6 boulangeries sur 5 produits, calculées à partir d'un tableau de rang 2, puis arrondies au millième. Avec ta fonction :
- a) `rank_factorization(M_1)`, où `M_1` est le tableau des boulangeries de MB4.1 (cellule de données) : la matrice $\mathbf{F}$ (à comparer à ta copie) ;
- b) le rang trouvé pour `M_8` avec `tol=1e-10`, puis avec `tol=1e-2` (une liste de 2 entiers) ;
- c) avec `tol=1e-2`, le plus grand écart $\lvert(\mathbf{C}\mathbf{F})_{ij} - (\mathbf{M}_8)_{ij}\rvert$ (un nombre).

In [ ]:
M_8 = np.array([[8.286, 12.286, 4.0, 2.571, 4.857], [9.0, 13.571, 4.286, 2.714, 5.429], [7.0, 11.286, 3.143, 1.857, 4.714],
 [6.0, 6.857, 3.429, 2.571, 2.143], [10.286, 14.571, 5.143, 3.429, 5.571], [8.571, 8.857, 5.143, 4.0, 2.429]])   # the mean sales of a week (6 bakeries x 5 products), from a table of rank 2, rounded
M_8.setflags(write=False)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB4")  # reload your saved file

**Les tests** de `rank_factorization` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB4, "test_rank_factorization_ or test_module_uses_")

**L'usage** : le tableau des boulangeries, puis un tableau mesuré et deux tolérances.

In [ ]:
answer_MB4_8a = answer_MB4_8b = answer_MB4_8c = ...
with wbm.attempt("MB4.8"):          # if you call rank_factorization here: ⏳ while it is not written
    answer_MB4_8a = ...  # a) a 2 x 5 matrix, as computed: the F of rank_factorization(M_1)
    answer_MB4_8b = ...  # b) a list of 2 integers: the rank found for M_8 with tol=1e-10, then with tol=1e-2
    answer_MB4_8c = ...  # c) a number, as computed: with tol=1e-2, the largest gap |(C F)_ij - (M_8)_ij|

In [ ]:
for letter, answer in zip("abc", [answer_MB4_8a, answer_MB4_8b, answer_MB4_8c]):
    check_answer(f"MB4.8{letter}", answer)

**Ce qu'il faut regarder** : pourquoi la tolérance $10^{-10}$ trouve-t-elle un autre rang ? D'où vient l'écart de c), et quel ordre de grandeur pouvait-on prévoir ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB4.9 — Les points de Borda de classements complets, et le MRR exact d'un ordre tiré au hasard, en fractions · `borda_scores`, `random_ranking_mrr` 🔨 ★★ ⏱️ 25 min
**Objectif :** programmer les points de Borda de classements complets et le MRR exact d'un ordre tiré au hasard, puis retrouver MB4.4 et la valeur $H_n/n$.  
**Prérequis :** fiche §5 et §6 ; R8 (combinaisons) ; Ex MB4.4 · **mymath :** `ranks.py` · **Parcours :** C

> Même mode d'emploi qu'en MB4.8 : écris les fonctions dans ta copie `mon_travail/mymath/ranks.py`, **enregistre**, puis exécute les cellules ci-dessous ; la première recharge ta librairie.

Écris les deux fonctions, en suivant leurs docstrings : pour `borda_scores(rankings)`, contrôle le tableau (2-D, non vide, des entiers, chaque ligne une permutation de $0, \ldots, n - 1$), puis donne au candidat placé à la position $p$ d'une ligne (positions à partir de 0) $n - 1 - p$ points, et additionne sur les lignes ; pour `random_ranking_mrr(n, n_relevant=1, depth=None)`, contrôle les trois arguments (des entiers, pas des booléens, chacun dans son domaine), puis additionne $P(R = j)/j$ en fractions, pour les rangs $j$ qui comptent. Lance les deux groupes de tests.

**L'usage.** La cellule de données de la partie A donne `RANKINGS_4`, les classements des trois jurys de MB4.4 (les pâtissiers A à E numérotés de 0 à 4) ; le notebook fournit `RANKINGS_9`, les classements de 7 juges qui classent 9 candidats. Avec tes fonctions :
- a) les points de Borda de `RANKINGS_4` (une liste de 5 entiers, à comparer à ta copie) ;
- b) les points de Borda de `RANKINGS_9` (une liste de 9 entiers) ;
- c) `random_ranking_mrr(8)`, `random_ranking_mrr(8, n_relevant=3)` et `random_ranking_mrr(12, depth=3)` (une liste de 3 fractions, des `Fraction` ou des chaînes `"p/q"`) ;
- d) vrai ou faux : `random_ranking_mrr(n)` est égal à $H_n/n$, en fractions exactes, pour $n = 1$ à 40.

In [ ]:
RANKINGS_9 = np.array([[4, 7, 1, 0, 2, 3, 6, 8, 5], [0, 7, 3, 8, 2, 6, 5, 4, 1], [5, 4, 7, 3, 6, 0, 8, 1, 2],
 [7, 1, 4, 2, 0, 8, 5, 3, 6], [3, 6, 5, 7, 1, 0, 4, 2, 8], [1, 6, 8, 2, 7, 4, 3, 5, 0],
 [8, 4, 3, 7, 5, 2, 6, 0, 1]])   # 7 judges rank 9 candidates (0 to 8), best first
RANKINGS_9.setflags(write=False)

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="MB4")  # reload your saved file

**Les tests** de `borda_scores`, puis ceux de `random_ranking_mrr` (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group(TEST_MB4, "test_borda_scores_")

In [ ]:
run_pytest_group(TEST_MB4, "test_random_ranking_mrr_")

**L'usage** : les trois jurys de MB4.4, sept juges, et trois ordres tirés au hasard.

In [ ]:
answer_MB4_9a = answer_MB4_9b = answer_MB4_9c = answer_MB4_9d = ...
with wbm.attempt("MB4.9"):          # if you call your functions here: ⏳ while they are not written
    answer_MB4_9a = ...  # a) a list of 5 integers: the Borda points of RANKINGS_4 (A to E)
    answer_MB4_9b = ...  # b) a list of 9 integers: the Borda points of RANKINGS_9
    answer_MB4_9c = ...  # c) a list of 3 fractions (Fraction, or strings "p/q"): the three calls of the statement
    answer_MB4_9d = ...  # d) True or False: random_ranking_mrr(n) equals H_n / n exactly, for n = 1 to 40

In [ ]:
for letter, answer in zip("abcd", [answer_MB4_9a, answer_MB4_9b, answer_MB4_9c, answer_MB4_9d]):
    check_answer(f"MB4.9{letter}", answer)

**Ce qu'il faut regarder** : le classement des 9 candidats par leurs points de Borda est-il celui de la somme de leurs rangs ? Combien d'ordres faudrait-il énumérer pour c) avec $n = 12$, et combien de termes additionne ta fonction ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie B · Prédire, dessiner

Fiche §4 et §6. Le MRR d'un ordre tiré au hasard, prédit puis simulé ; les courbes de saturation et l'effet de la longueur d'une fiche.

### Ex MB4.10 — Prédire le MRR d'un ordre tiré au hasard parmi 10, 100 puis 1 000 candidats, puis le simuler et le comparer à H_n/n 🔮 ★ ⏱️ 15 min
**Objectif :** prévoir comment le MRR d'un ordre tiré au hasard décroît avec le nombre de candidats, puis le simuler et le comparer à $H_n/n$.  
**Prérequis :** fiche §6 ; M02a (rectangles), R8 ; Ex MB4.9 · **Parcours :** C

Un seul des $n$ candidats est pertinent, et l'ordre est tiré au hasard parmi les $n!$ ordres. **Avant d'exécuter quoi que ce soit**, prédis :
- a) pour $n = 10$, le MRR est le plus proche de : (A) 0,1 ; (B) 0,5 ; (C) 0,3 (une lettre) ;
- b) de $n = 100$ à $n = 1\,000$, le MRR est divisé par environ : (A) 7 ; (B) 10 ; (C) 3 (une lettre) ;
- c) de $n = 10$ à $n = 1\,000$, le produit $n \times \text{MRR}$ : (A) ne change presque pas ; (B) augmente de 4 à 5 ; (C) est multiplié par environ 100 (une lettre).

Puis, une fois tes prédictions écrites et vérifiées, simule : avec `rng = np.random.default_rng(7510)`, pour $n = 10$, 100 puis 1 000, dans cet ordre, tire 20 000 fois `order = rng.permutation(n)` ; le pertinent est l'objet 0, et son rang est `int(np.flatnonzero(order == 0)[0]) + 1`. Relève, chaque fois pour $n = 10$, 100 et 1 000 (une liste de 3 nombres) :
- d) la moyenne simulée de $1/R$ ;
- e) $H_n/n$ ;
- f) $\frac{\ln n}{n}$ ;
- g) vrai ou faux : chaque moyenne simulée est à moins de 3 erreurs types de $H_n/n$.

La vérification de l'expérience ne s'exécute qu'une fois tes trois prédictions écrites.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_MB4_10a = ...  # a) "A", "B" or "C": for n = 10, the MRR is closest to (A) 0.1, (B) 0.5, (C) 0.3
prediction_MB4_10b = ...  # b) "A", "B" or "C": from n = 100 to n = 1000, the MRR is divided by about (A) 7, (B) 10, (C) 3
prediction_MB4_10c = ...  # c) "A", "B" or "C": from n = 10 to n = 1000, n x MRR (A) hardly changes, (B) grows by 4 to 5, (C) is multiplied by about 100

In [ ]:
for letter, prediction in zip("abc", [prediction_MB4_10a, prediction_MB4_10b, prediction_MB4_10c]):
    wbm.check(f"MB4.10{letter}", prediction)

**L'expérience.** Écris-la une fois tes trois prédictions vérifiées ; sa vérification ne s'exécute qu'à cette condition.

In [ ]:
answer_MB4_10d = answer_MB4_10e = answer_MB4_10f = answer_MB4_10g = ...
with wbm.attempt("MB4.10"):          # your simulation
    rng = np.random.default_rng(7510)   # one generator, created once, for n = 10, 100, 1000 in this order
    answer_MB4_10d = ...  # d) a list of 3 numbers, as computed: the simulated mean of 1/R for n = 10, 100, 1000
    answer_MB4_10e = ...  # e) a list of 3 numbers, as computed: H_n / n for n = 10, 100, 1000
    answer_MB4_10f = ...  # f) a list of 3 numbers, as computed: ln(n) / n for n = 10, 100, 1000
    answer_MB4_10g = ...  # g) True or False: each simulated mean is within 3 standard errors of H_n / n

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_MB4_10a, prediction_MB4_10b, prediction_MB4_10c]):
    print("⏳ Écris d'abord tes trois prédictions, puis exécute à nouveau.")
else:
    def hint_10d(value):
        """A targeted hint for d): H_n / n instead of the simulation, or plausible means of another draw."""
        if passes("MB4.10e", value):
            return "d) est la moyenne simulée de 1/R, pas H_n/n : relève la moyenne de tes 20 000 valeurs"
        values = as_floats(value)
        if values is None or values.shape != (3,):
            return None
        plausible = []
        for got, n in zip(values, (10, 100, 1000)):
            mean = float(ref_ranks.random_ranking_mrr(n))                  # the exact value, by the reference
            variance = sum(1 / k ** 2 for k in range(1, n + 1)) / n - mean ** 2
            plausible.append(abs(got - mean) <= 4 * math.sqrt(variance / 20000))
        if all(plausible):
            return ("tes moyennes sont plausibles, mais ce ne sont pas celles du tirage demandé : un seul générateur "
                    "np.random.default_rng(7510), créé une fois, puis n = 10, 100 et 1 000 dans cet ordre, 20 000 fois "
                    "rng.permutation(n) pour chacun, et le rang int(np.flatnonzero(order == 0)[0]) + 1")
        return None


    check_answer("MB4.10d", answer_MB4_10d, hint=hint_10d)
    check_answer("MB4.10e", answer_MB4_10e)
    check_answer("MB4.10f", answer_MB4_10f)
    check_answer("MB4.10g", answer_MB4_10g)

**Ce qu'il faut regarder** (après l'expérience) : pourquoi le MRR n'est-il pas divisé par 10 quand $n$ est multiplié par 10 ? Quelle ligne de base un classement doit-il battre parmi 1 000 candidats ? Les réponses sont dans `05_solutions.md`, section Notebook.

### Ex MB4.11 — Les courbes de saturation x ↦ x(k₁ + 1)/(x + k₁) de k₁ = 0 à l'infini, et ce que la normalisation par la longueur leur fait 🎨 ★ ⏱️ 15 min
**Objectif :** tracer la saturation pour plusieurs $k_1$, entre la seule présence et le compte brut, et voir ce que la normalisation par la longueur lui fait.  
**Prérequis :** fiche §4 ; R7, M05a (concavité) ; Ex MB4.3 · **Parcours :** C

Écris `saturation_11(x, k1)`, qui rend $s(x) = \frac{x(k_1 + 1)}{x + k_1}$ pour un tableau `x` de nombres $\ge 0$ (avec $k_1 = 0$, la seule présence : 1 pour $x > 0$, 0 pour $x = 0$ ; avec `k1=np.inf`, le compte brut $x$) ; puis `draw_saturations_11(ax, k1s, xmax)`, qui trace dans les axes reçus, sur 241 points de $[0, x_{\max}]$ (`np.linspace(0, xmax, 241)`), une courbe pour chaque $k_1$ de `k1s`, avec une légende qui nomme chaque $k_1$ et des noms d'axes, et rend la liste des tableaux tracés ; puis `draw_lengths_11(ax, k1, b, ells, xmax)`, qui trace de même, pour chaque $\ell$ de `ells`, la courbe de la saturation normalisée $x \mapsto \frac{x(k_1 + 1)}{x + k_1\nu}$, avec $\nu = 1 - b + b\ell$, avec une légende qui nomme chaque $\ell$. Les fonctions dessinent dans les axes reçus, sans ouvrir de figure.

La vérification les appelle avec $k_1 = 0$, 1, 2, 3 et l'infini, puis avec $k_1 = 1$, $b = \frac12$ et $\ell = \frac12$, 1, 2, sur $[0, 12]$, affiche la figure, et les essaie sur des appels cachés. Relève :
- a) $s(6)$ pour $k_1 = 1$, 2 et 3 (une liste de 3 nombres) ;
- b) pour $k_1 = 1$, 2 et 3, la valeur de $x$ où $s$ atteint 90 % de sa limite (une liste de 3 nombres) ;
- c) pour $k_1 = 1$ et $b = \frac12$, la valeur en $x = 4$ pour $\ell = \frac12$, 1 et 2 (une liste de 3 nombres) ;
- d) vrai ou faux : pour $\ell = 1$, toutes les courbes passent par le point $(1, 1)$.

In [ ]:
def saturation_11(x, k1):
    """s(x) = x (k1 + 1) / (x + k1) for an array x of numbers >= 0: with k1 = 0, the presence alone (1 for x > 0, 0 for
    x = 0); with k1 = np.inf, the raw count x. Returns an array of the shape of x."""
    raise NotImplementedError("saturation_11 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_saturations_11(ax, k1s, xmax):
    """In the axes ax: for each k1 of k1s, the curve of saturation_11 on the 241 points np.linspace(0, xmax, 241), with
    a legend that names each k1 and axis labels. Returns the list of the arrays drawn (one per k1). Opens no figure."""
    raise NotImplementedError("draw_saturations_11 n'est pas encore écrite (04_indices.md si tu bloques)")


def draw_lengths_11(ax, k1, b, ells, xmax):
    """In the axes ax: for each length ell of ells (relative to the mean length), the curve of
    x -> x (k1 + 1) / (x + k1 nu), nu = 1 - b + b ell, on np.linspace(0, xmax, 241), with a legend that names each ell
    and axis labels. Returns the list of the arrays drawn (one per ell). Opens no figure."""
    raise NotImplementedError("draw_lengths_11 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB4_11a = answer_MB4_11b = answer_MB4_11c = answer_MB4_11d = ...
with wbm.attempt("MB4.11"):          # your functions: ⏳ while they are not written
    answer_MB4_11a = ...  # a) a list of 3 numbers, as computed: s(6) for k1 = 1, 2, 3
    answer_MB4_11b = ...  # b) a list of 3 numbers: for k1 = 1, 2, 3, the x where s reaches 90 % of its limit
    answer_MB4_11c = ...  # c) a list of 3 numbers, as computed: k1 = 1, b = 0.5, the value at x = 4 for ell = 0.5, 1, 2
    answer_MB4_11d = ...  # d) True or False: for ell = 1, every curve passes through the point (1, 1)

In [ ]:
for letter, answer in zip("abcd", [answer_MB4_11a, answer_MB4_11b, answer_MB4_11c, answer_MB4_11d]):
    check_answer(f"MB4.11{letter}", answer)

HIDDEN_C11 = {'x': [0.0, 0.4, 1.0, 2.5, 7.0, 31.0],
 'k1': [0, 0.7, 2.5, math.inf],
 'id_s': 'MB4.11e',
 'draw_k1': [0.7, 2.5, math.inf],
 'draw_xmax': 9,
 'id_l': 'MB4.11f',
 'k1_l': 2,
 'b_l': 0.25,
 'ells': [0.6, 3],
 'xmax_l': 15,
 'id_h': 'MB4.11g',
 'idx': [0, 8, 16, 24, 32, 40, 56, 72, 80, 88, 104, 112, 120, 128, 144, 152, 160, 168, 176, 184, 192, 200, 208,
         216, 224, 240]}   # hidden arguments, and the sub-IDs of the values expected

_NAMES_C11 = r"(?:k_?\{?1\}?|k₁|k1|\\nu|ν|nu|\bb)"


def label_value_c11(label, drop):
    """The first number written in a label of the legend (½ is 0.5, ∞ or inf is the infinity), after the names k₁ and,
    for drop=True, the parameters k₁ = …, b = … and ν = … with their values are set aside; None when there is none."""
    text = str(label)
    if not text or text.startswith("_"):
        return None
    text = re.sub(r"\\frac\{(\d+)\}\{(\d+)\}", r" \1/\2 ", text)
    text = text.replace("½", " 0.5 ").replace("¼", " 0.25 ").replace("¾", " 0.75 ").replace(",", ".")
    if drop:
        text = re.sub(_NAMES_C11 + r"\s*(?:=|:)\s*[-+]?[\d.]+(?:/\d+)?", " ", text)
    text = re.sub(r"k_?\{?1\}?|k₁|k1", " ", text)
    match = re.search(r"(∞|\binf(?:ty|ini|inity)?\b)|(?<![\w.])(\d+(?:\.\d+)?)(?:/(\d+))?", text,
                      flags=re.IGNORECASE)
    if match is None:
        return None
    if match.group(1):
        return math.inf
    return float(match.group(2)) / float(match.group(3) or 1)


def legend_names_c11(ax, grid, curves, values, drop):
    """True when each curve (drawn at grid) is a line of ax whose label names its value first."""
    for curve, value in zip(curves, values):
        found = False
        for line in ax.lines:
            x, y = np.asarray(line.get_xdata(), dtype=float), np.asarray(line.get_ydata(), dtype=float)
            if x.shape == grid.shape and y.shape == grid.shape and np.allclose(x, grid) \
                    and np.allclose(y, curve, equal_nan=True):
                named = label_value_c11(line.get_label(), drop)
                found = found or (named is not None and (named == value or math.isclose(named, value)))
        if not found:
            return False
    return True


def curves_why_c11(ax, values, xmax, curves, right, what, name):
    """What is wrong in a call of a drawing function (French phrases); right(curves) tells whether the curves returned
    are the right ones."""
    if not isinstance(curves, (list, tuple)) or len(curves) != len(values):
        return [f"{what} doit rendre la liste des {len(values)} tableaux tracés (un par valeur), reçu {shown(curves)}"]
    grid = np.linspace(0, xmax, 241)
    curves = [as_floats(c) for c in curves]
    if any(c is None or c.shape != grid.shape for c in curves):
        return [f"chaque tableau rendu par {what} doit avoir 241 valeurs, une par point de np.linspace(0, xmax, 241)"]
    problems = []
    if not right(curves):
        problems.append(f"les tableaux rendus doivent être les courbes de chaque {name} reçu, dans l'ordre reçu")
    if not all(line_with(ax, grid, c) for c in curves):
        problems.append("chaque courbe doit être tracée en fonction des 241 points de [0, xmax], np.linspace(0, xmax, 241)")
    if ax.get_legend() is None:
        problems.append("il faut une légende (label=…, puis ax.legend())")
    elif not legend_names_c11(ax, grid, curves, values, drop=(name == "ℓ")):
        problems.append(f"la légende doit nommer le {name} de chaque courbe (label=f\"{name} = {{valeur}}\")")
    if not ax.get_xlabel() or not ax.get_ylabel():
        problems.append("il faut des noms d'axes")
    return problems


with checked("MB4.11"):
    results_11 = []
    report_11 = reporter(results_11, stop=False)
    x_c11 = np.array(HIDDEN_C11["x"])                          # non-dyadic floats: a change in place would show
    got_c11, changed_c11 = [], False
    for k1_c11 in HIDDEN_C11["k1"]:
        received_c11 = x_c11.copy()
        got_c11.append(as_floats(call_safely("Ex MB4.11 : saturation_11 (appel caché)", saturation_11, received_c11,
                                             k1_c11)))
        changed_c11 = changed_c11 or not np.array_equal(received_c11, x_c11)
    if changed_c11:
        report_11(False, "", "saturation_11 modifie le tableau x reçu : calcule sur une copie")
    elif any(g is None or g.shape != x_c11.shape for g in got_c11):
        report_11(False, "", "saturation_11 rend un tableau de la forme de x")
    else:
        report_11(passes(HIDDEN_C11["id_s"], [g.tolist() for g in got_c11]),
                  "saturation_11 : des appels cachés (d'autres x, d'autres k₁, dont 0 et l'infini) sont justes",
                  "saturation_11 sur des appels cachés : ce n'est pas s(x) = x(k₁ + 1)/(x + k₁) ; relis les deux cas à "
                  "part, k₁ = 0 (la seule présence) et k₁ infini (le compte brut)")

    def saturations_right_c11(k1s, xmax):
        grid = np.linspace(0, xmax, 241)
        return lambda curves: all(matches(c, saturation_11(grid.copy(), k1), rtol=1e-9, atol=1e-12)
                                  for c, k1 in zip(curves, k1s))

    def lengths_right_c11(sub_id):
        return lambda curves: passes(sub_id, [c[HIDDEN_C11["idx"]].tolist() for c in curves])

    fig_11, axes_11 = plt.subplots(1, 2, figsize=(12, 4.5))
    try:
        drawn_11 = []
        opened_11 = drawn(lambda: drawn_11.append(call_safely("Ex MB4.11 : draw_saturations_11", draw_saturations_11,
                                                              axes_11[0], (0, 1, 2, 3, np.inf), 12)))
        problems_11 = curves_why_c11(axes_11[0], (0, 1, 2, 3, math.inf), 12, drawn_11[0],
                                     saturations_right_c11((0, 1, 2, 3, np.inf), 12), "draw_saturations_11", "k₁")
        report_11(not problems_11, "draw_saturations_11 (k₁ = 0, 1, 2, 3 et l'infini, sur [0, 12]) : les courbes, la "
                  "légende et les noms d'axes", "draw_saturations_11 : " + " ; ".join(problems_11))
        lengths_11 = []
        opened_11 += drawn(lambda: lengths_11.append(call_safely("Ex MB4.11 : draw_lengths_11", draw_lengths_11,
                                                                 axes_11[1], 1, 0.5, (0.5, 1, 2), 12)))
        problems_11 = curves_why_c11(axes_11[1], (0.5, 1, 2), 12, lengths_11[0], lengths_right_c11(HIDDEN_C11["id_l"]),
                                     "draw_lengths_11", "ℓ")
        report_11(not problems_11, "draw_lengths_11 (k₁ = 1, b = ½, ℓ = ½, 1 et 2, sur [0, 12]) : les courbes, la "
                  "légende et les noms d'axes", "draw_lengths_11 : " + " ; ".join(problems_11))
        fig_c11, axes_c11 = plt.subplots(1, 3)          # the 3rd axes is the current one: plt.plot would draw there
        try:
            hidden_s_c11, hidden_l_c11 = [], []
            opened_11 += drawn(lambda: hidden_s_c11.append(call_safely(
                "Ex MB4.11 : draw_saturations_11 (appel caché)", draw_saturations_11, axes_c11[0],
                tuple(HIDDEN_C11["draw_k1"]), HIDDEN_C11["draw_xmax"])))
            problems_c11 = curves_why_c11(axes_c11[0], HIDDEN_C11["draw_k1"], HIDDEN_C11["draw_xmax"], hidden_s_c11[0],
                                          saturations_right_c11(HIDDEN_C11["draw_k1"], HIDDEN_C11["draw_xmax"]),
                                          "draw_saturations_11", "k₁")
            opened_11 += drawn(lambda: hidden_l_c11.append(call_safely(
                "Ex MB4.11 : draw_lengths_11 (appel caché)", draw_lengths_11, axes_c11[1], HIDDEN_C11["k1_l"],
                HIDDEN_C11["b_l"], tuple(HIDDEN_C11["ells"]), HIDDEN_C11["xmax_l"])))
            problems_c11 += curves_why_c11(axes_c11[1], HIDDEN_C11["ells"], HIDDEN_C11["xmax_l"], hidden_l_c11[0],
                                           lengths_right_c11(HIDDEN_C11["id_h"]), "draw_lengths_11", "ℓ")
            elsewhere_11 = bool(axes_c11[2].lines or axes_c11[2].collections or axes_c11[2].get_legend())
        finally:
            plt.close(fig_c11)
        report_11(not problems_c11, "des appels cachés (d'autres k₁, d'autres longueurs, un autre b) sont justes aussi",
                  "des appels cachés (d'autres k₁, d'autres longueurs, un autre b) : "
                  + " ; ".join(dict.fromkeys(problems_c11)))
        report_11(not opened_11 and not elsewhere_11, "tes fonctions dessinent dans les axes reçus",
                  "tes fonctions doivent dessiner dans les axes reçus (ax.plot, ax.legend…), sans ouvrir de figure ni "
                  "dessiner dans l'axe courant (plt.plot, plt.legend…)")
    except BaseException:
        plt.close(fig_11)
        raise
    fig_11.tight_layout()
    plt.show()
    verdict("MB4.11", all(results_11) and len(results_11) == 5, "ta saturation et tes deux figures sont justes, sur les "
            "appels du notebook et sur des appels cachés.", "relis les lignes ❌ ci-dessus.")

**Ce qu'il faut regarder** : entre quelles deux courbes extrêmes se placent les autres ? Que fait une fiche longue à sa courbe, et une fiche courte ? Les réponses sont dans `05_solutions.md`, section Notebook.

## Partie C · Déboguer

Fiche §1, §3 et §5. Trois erreurs classiques à corriger, qui passent une vérification mal choisie.

### Ex MB4.12 — Chasse à l'erreur : C pris dans la forme réduite au lieu de M, un classement lu comme la liste des rangs des candidats, un poids de rareté écrit ln(n/N) 🐛 ★ ⏱️ 15 min
**Objectif :** reconnaître trois erreurs classiques (C pris dans la forme réduite, un classement lu comme la liste des rangs, un poids de rareté de signe faux), qui passent une vérification mal choisie, puis les corriger.  
**Prérequis :** fiche §1, §3 et §5 ; Ex MB4.1, MB4.3 et MB4.4 · **Parcours :** C

Un collègue a écrit trois fonctions, chacune accompagnée d'une vérification qui passe (cellule fournie) :
1. `rank_factorization_bad(M)` doit rendre la factorisation par les colonnes pivots ; elle prend $\mathbf{C}$ dans les colonnes pivots de la **forme réduite** au lieu de celles de $\mathbf{M}$. Il l'a vérifiée sur une matrice déjà sous forme échelonnée réduite.
2. `borda_bad(rankings)` doit rendre les points de Borda ; elle lit chaque ligne comme la liste des **rangs** des candidats (la valeur d'indice $c$ comme la position du candidat $c$), au lieu de la liste des candidats dans l'ordre. Il l'a vérifiée sur trois jurys, `[[1, 0, 2], [0, 2, 1], [2, 1, 0]]`.
3. `rarity_bad(n, N)` doit rendre la surprise d'un ingrédient présent dans $n$ fiches sur $N$ ; elle rend $\ln\frac nN$. Il l'a vérifiée sur un ingrédient présent dans toutes les fiches, de poids nul.

À faire : trouve et corrige chaque erreur ; les fonctions corrigées s'appellent `rank_factorization_fixed`, `borda_fixed` et `rarity_fixed`, avec les mêmes arguments (sans `mymath` ; `exact_rref_12` de la cellule fournie reste permise). Le notebook fournit `RANK_12`, les classements de 4 juges et 6 candidats. Puis réponds :
- a) pour le tableau `M_1` des boulangeries, le plus grand écart $\lvert(\mathbf{C}\mathbf{F})_{ij} - M_{ij}\rvert$ par la fonction corrigée, puis par la fonction fautive (une liste de 2 nombres) ;
- b) pour `RANK_12`, les points de Borda par la fonction corrigée, puis par la fonction fautive (une liste de 2 listes de 6 entiers) ;
- c) les surprises d'ingrédients présents dans 3 et dans 30 fiches sur 90, par la fonction corrigée, puis par la fonction fautive (une liste de 4 nombres).

Dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait pas. La vérification essaie aussi tes fonctions corrigées sur des cas cachés : d'autres tableaux, dont des tableaux de rang 1 et des tableaux dont les colonnes pivots ne sont pas les premières, des classements dont aucune ligne n'est sa propre inverse, et d'autres carnets.

In [ ]:
import sympy as sp

RANK_12 = np.array([[4, 0, 1, 2, 3, 5], [2, 4, 5, 0, 1, 3], [4, 0, 5, 1, 2, 3], [0, 4, 1, 2, 3, 5]])   # 4 judges rank 6 candidates (0 to 5), best first
RANK_12.setflags(write=False)


# The colleague's code
def exact_rref_12(M):
    """The reduced row echelon form of M, computed exactly (SymPy, the entries read as decimal fractions), as an array
    of floats, and the list of its pivot columns."""
    R, pivots = sp.Matrix([[sp.Rational(str(x)) for x in row] for row in np.asarray(M, dtype=float)]).rref()
    return np.array(R, dtype=float), list(pivots)


def rank_factorization_bad(M):
    """The rank factorization (C, F) of M: C the pivot columns of M, F the non-zero rows of its reduced form."""
    R, pivots = exact_rref_12(M)
    return R[:, pivots], R[:len(pivots)]


def borda_bad(rankings):
    """The Borda points of complete rankings (each row lists the candidates, best first)."""
    R = np.asarray(rankings)
    m, n = R.shape
    points = np.zeros(n, dtype=int)
    for row in R:
        for c in range(n):
            points[c] += n - 1 - row[c]
    return points


def rarity_bad(n, N):
    """The surprise of an ingredient found in n recipe cards out of N."""
    return math.log(n / N)


# The colleague's checks: all pass
E_12 = np.array([[1.0, 0, 5, 0, -2], [0, 1, 3, 0, 4], [0, 0, 0, 1, 7]])   # a matrix already in reduced row echelon form
C_12, F_12 = rank_factorization_bad(E_12)
print("(1)", bool(np.allclose(C_12 @ F_12, E_12)), "(attendu True : C F redonne la matrice)")
print("(2)", borda_bad([[1, 0, 2], [0, 2, 1], [2, 1, 0]]).tolist() == [3] * 3,
      "(attendu True : trois jurys, trois points chacun, à la main)")
print("(3)", rarity_bad(90, 90) == 0, "(attendu True : un ingrédient présent dans toutes les fiches ne surprend pas)")

In [ ]:
def rank_factorization_fixed(M):
    """The rank factorization (C, F) of M: C the pivot columns of M, F the non-zero rows of its reduced form."""
    raise NotImplementedError("rank_factorization_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def borda_fixed(rankings):
    """The Borda points of complete rankings (each row lists the candidates, best first)."""
    raise NotImplementedError("borda_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


def rarity_fixed(n, N):
    """The surprise of an ingredient found in n recipe cards out of N."""
    raise NotImplementedError("rarity_fixed n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB4_12a = answer_MB4_12b = answer_MB4_12c = ...
with wbm.attempt("MB4.12"):          # your three functions: ⏳ while they are not written
    answer_MB4_12a = ...  # a) a list of 2 numbers, as computed: max |CF - M_1| by rank_factorization_fixed, then by the WRONG rank_factorization_bad
    answer_MB4_12b = ...  # b) a list of 2 lists of 6 integers: borda_fixed(RANK_12), then the WRONG borda_bad(RANK_12)
    answer_MB4_12c = ...  # c) a list of 4 numbers, as computed: rarity_fixed for 3 and 30 cards out of 90, then the WRONG rarity_bad

In [ ]:
for letter, answer in zip("abc", [answer_MB4_12a, answer_MB4_12b, answer_MB4_12c]):
    check_answer(f"MB4.12{letter}", answer)

CASES_C12 = {'tables': [[[0.0, 0.3, 0.6, -0.9], [0.0, 0.7, 1.4, -2.1], [0.0, -1.1, -2.2, 3.3]],
            [[1.3, -1.95, 0.4, 2.1, 0.25], [0.2, -0.3, 1.1, 2.4, -1.0], [-0.7, 1.05, 0.9, 1.1, -1.25],
             [2.1, -3.15, -0.6, 0.9, 1.65]],
            [[0.9, 0.5, 0.4, -0.3], [-0.4, 1.6, -2.0, 0.6], [1.7, -0.8, 2.5, 1.4], [0.3, 2.2, -1.9, -0.5],
             [-1.2, 0.1, -1.3, 0.8]]],
 'rankings': [[[5, 3, 0, 2, 1, 6, 4], [2, 3, 1, 6, 4, 5, 0], [0, 1, 5, 6, 3, 4, 2], [0, 3, 5, 1, 6, 4, 2],
               [4, 3, 2, 5, 0, 6, 1]],
              [[3, 2, 4, 1, 0], [4, 1, 2, 0, 3], [2, 3, 4, 1, 0]],
              [[3, 1, 5, 6, 7, 2, 0, 4], [3, 6, 0, 7, 5, 2, 1, 4], [4, 3, 1, 2, 6, 7, 0, 5],
               [2, 6, 1, 5, 7, 4, 0, 3], [7, 2, 6, 0, 5, 3, 1, 4], [7, 2, 4, 6, 3, 5, 0, 1]]],
 'notebooks': [[7, 120], [33, 61], [250, 400], [1, 13]],
 'id': 'MB4.12d'}   # the hidden tables, rankings and notebooks of recipes, and the sub-ID expected

results_12, parts_12 = [], []


def report_12(ok, success, failure):
    results_12.append(bool(ok))
    print(("✅ " + success) if ok else ("❌ " + failure))


def call_12(part, name, function, *args):
    """function(*args); an error of the learner's code becomes one ❌ line that names the function."""
    try:
        return function(*args)
    except (NotImplementedError, MymathMissingError):
        raise
    except Exception as exc:  # noqa: BLE001 - the learner's error, reported with the name of the function
        report_12(False, "", f"({part}) {name} : ton code lève {type(exc).__name__} : {str(exc)[:160]}")
        parts_12.append(part)
        raise StopCheck from None


def factors_12(got):
    """(C, F) as two 2-D float arrays, or None when got is not such a pair."""
    try:
        C, F = got
        C, F = np.asarray(C, dtype=float), np.asarray(F, dtype=float)
    except (TypeError, ValueError):
        return None
    return (C, F) if C.ndim == 2 and F.ndim == 2 else None


with checked("MB4.12"):                                      # (1) rank_factorization_fixed
    wrong_c12 = []
    for table_c12 in CASES_C12["tables"]:
        M_c12 = np.array(table_c12)                          # non-dyadic floats: a change in place would show
        before_c12 = M_c12.copy()
        got_c12 = factors_12(call_12(1, "rank_factorization_fixed", rank_factorization_fixed, M_c12))
        want_c12 = ref_ranks.rank_factorization(before_c12)
        if not np.array_equal(M_c12, before_c12):
            wrong_c12.append("ta fonction modifie le tableau reçu : travaille sur une copie")
        elif got_c12 is None:
            wrong_c12.append("rank_factorization_fixed rend le couple (C, F) de deux tableaux 2-D")
        elif all(matches(g, w, rtol=1e-9, atol=1e-9) for g, w in zip(got_c12, want_c12)):
            continue
        elif matches(got_c12[0], rank_factorization_bad(before_c12)[0], atol=1e-9):
            wrong_c12.append("ta correction prend encore C dans la forme réduite : C est faite des colonnes pivots de M "
                             "elle-même")
        elif got_c12[0].shape == want_c12[0].shape and matches(got_c12[0], before_c12[:, :got_c12[0].shape[1]]):
            wrong_c12.append("sur des tableaux dont les colonnes pivots ne sont pas les premières, C n'est pas faite des "
                             "colonnes pivots de M")
        else:
            wrong_c12.append("sur d'autres tableaux (de rang 1, des colonnes pivots qui ne sont pas les premières), ce "
                             "n'est pas la factorisation par les colonnes pivots : C les colonnes pivots de M, F les "
                             "lignes non nulles de sa forme réduite")
    report_12(not wrong_c12, "(1) rank_factorization_fixed : des tableaux cachés, de rang 1 ou de colonnes pivots qui ne "
              "sont pas les premières", "(1) rank_factorization_fixed : " + " ; ".join(dict.fromkeys(wrong_c12)))
    parts_12.append(1)

with checked("MB4.12"):                                      # (2) borda_fixed, compared with the reference
    wrong_c12 = []
    for rankings_c12 in CASES_C12["rankings"]:
        R_c12 = np.array(rankings_c12)
        before_c12 = R_c12.copy()
        got_c12 = as_floats(call_12(2, "borda_fixed", borda_fixed, R_c12))
        want_c12 = ref_ranks.borda_scores(before_c12)
        if not np.array_equal(R_c12, before_c12):
            wrong_c12.append("ta fonction modifie le tableau reçu : travaille sur une copie")
        elif got_c12 is None or got_c12.shape != want_c12.shape:
            wrong_c12.append(f"borda_fixed rend un tableau de {before_c12.shape[1]} points, un par candidat")
        elif np.array_equal(got_c12, want_c12):
            continue
        elif np.array_equal(got_c12, borda_bad(before_c12)):
            wrong_c12.append("ta correction lit encore chaque ligne comme la liste des rangs des candidats : une ligne "
                             "est un ordre, le candidat placé à chaque position")
        else:
            wrong_c12.append("sur des classements dont aucune ligne n'est sa propre inverse, ce ne sont pas les points de "
                             "Borda : le candidat placé à la position p d'une ligne (à partir de 0) reçoit n − 1 − p "
                             "points")
    report_12(not wrong_c12, "(2) borda_fixed : des classements cachés, dont aucune ligne n'est sa propre inverse",
              "(2) borda_fixed : " + " ; ".join(dict.fromkeys(wrong_c12)))
    parts_12.append(2)

with checked("MB4.12"):                                      # (3) rarity_fixed
    got_c12 = [call_12(3, "rarity_fixed", rarity_fixed, n_c12, N_c12) for n_c12, N_c12 in CASES_C12["notebooks"]]
    numbers_c12 = as_floats(got_c12)
    if numbers_c12 is None or numbers_c12.shape != (len(got_c12),):
        why_c12 = f"rarity_fixed rend un nombre, reçu {shown(got_c12)}"
    elif passes(CASES_C12["id"], numbers_c12.tolist()):
        why_c12 = ""
    elif all(math.isclose(g, rarity_bad(n, N), abs_tol=1e-9) for g, (n, N) in zip(numbers_c12, CASES_C12["notebooks"])):
        why_c12 = "ta correction rend encore ln(n/N), négatif pour un ingrédient rare"
    else:
        why_c12 = ("sur d'autres carnets, ce n'est pas la surprise d'un ingrédient présent dans n fiches sur N, un nombre "
                   "positif, en nats")
    report_12(not why_c12, "(3) rarity_fixed : d'autres carnets", f"(3) rarity_fixed : {why_c12}")
    parts_12.append(3)

if len(set(parts_12)) == 3:
    verdict("MB4.12", all(results_12), "les trois fonctions corrigées passent tous les cas.", "relis les lignes ❌ ci-dessus.")

**Après la vérification**, dans ta copie : pour chacune des trois erreurs, pourquoi la vérification du collègue ne la voyait-elle pas, et quelle vérification l'aurait vue ?

## Partie D · Défi

Fiche §2. Le rang caché de tableaux bruités, retrouvé par leurs valeurs singulières, vérifié sur des tableaux cachés tirés à chaque exécution.

### Ex MB4.13 — Défi : retrouver le rang caché de cinq tableaux bruités, cinq fois sur cinq, par leurs valeurs singulières 🏆 ★★★ ⏱️ 25 min
**Objectif :** retrouver le rang caché d'un tableau bruité en comparant ses valeurs singulières au niveau du bruit, estimé sur le tableau lui-même, et le vérifier sur des tableaux cachés.  
**Prérequis :** fiche §2 ; M12d (SVD, Eckart-Young) ; Ex MB4.2 · **Parcours :** C

Le notebook fournit `TABLES_13`, cinq tableaux de mesures de formes différentes. Chacun est la somme d'un tableau de rang $r$ (caché) et d'un bruit : chaque case reçoit un bruit indépendant, de moyenne nulle et de même écart-type $\sigma$ dans tout le tableau ; $\sigma$ change d'un tableau à l'autre, et n'est pas donné. Écris `hidden_rank_13(M)`, qui rend le rang caché d'un tableau (un entier $\ge 0$ : un tableau peut n'avoir aucun signal), en lisant ses valeurs singulières ; tout NumPy et SciPy sont permis (`np.linalg.svd(M, compute_uv=False)` donne les valeurs singulières).

**Critère de réussite** (les « cinq fois sur cinq » du titre) **:** le bon rang pour cinq tableaux cachés, tirés à nouveau à chaque exécution (l'un facile, l'un dont le signal dépasse à peine le bruit, l'un dont les valeurs singulières du signal s'étalent sur un facteur 10 ou plus, l'un très allongé, et l'un sans signal ou à une échelle extrême), en moins de 10 secondes en tout, et le bon rang pour les cinq tableaux du notebook : la vérification recalcule ta réponse a) avec ta fonction. Relève :
- a) les rangs que rend ta fonction pour les cinq tableaux (une liste de 5 entiers) ;
- b) `np.linalg.matrix_rank` de chacun (une liste de 5 entiers) ;
- c) le rang effectif à 99 % de chacun (une liste de 5 entiers).

Point de départ : le bruit seul a des valeurs singulières qui ne dépassent guère $\sigma(\sqrt m + \sqrt n)$ (fiche §2) ; $\sigma$ n'est pas donné, mais ce que le rang laisse de côté est presque tout du bruit, et tu sais ce que vaut son énergie (Eckart-Young).

In [ ]:
def _tables_13():
    """The five tables of the notebook, read in solutions/challenges/mb4_13.py (the hidden part of the challenge: the law
    of the tables and their hidden ranks stay there; reading it spoils the challenge)."""
    import importlib.util
    spec = importlib.util.spec_from_file_location("mb4_13_tables",
                                                  wbm.repo_root() / "solutions" / "challenges" / "mb4_13.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return [np.array(table, dtype=float) for table in module.notebook_tables()]


TABLES_13 = _tables_13()   # five noisy tables of measurements: shapes (90, 60), (70, 110), (120, 120), (80, 50), (100, 75)
del _tables_13
for _array in TABLES_13:
    _array.setflags(write=False)
print("formes :", [table.shape for table in TABLES_13])

In [ ]:
def hidden_rank_13(M):
    """The hidden rank of a noisy table M (an integer >= 0), read on its singular values."""
    raise NotImplementedError("hidden_rank_13 n'est pas encore écrite (04_indices.md si tu bloques)")


answer_MB4_13a = answer_MB4_13b = answer_MB4_13c = ...
with wbm.attempt("MB4.13"):          # your function: ⏳ while it is not written
    answer_MB4_13a = ...  # a) a list of 5 integers: hidden_rank_13 of each table of TABLES_13
    answer_MB4_13b = ...  # b) a list of 5 integers: np.linalg.matrix_rank of each table
    answer_MB4_13c = ...  # c) a list of 5 integers: the effective rank at 99 % of each table

In [ ]:
for letter, answer in zip("bc", [answer_MB4_13b, answer_MB4_13c]):
    check_answer(f"MB4.13{letter}", answer)

def check_c13():
    """The check of MB4.13, in a function: the hidden tables and their ranks stay local, and each hidden rank is read
    only after hidden_rank_13 has returned."""
    import importlib.util
    spec = importlib.util.spec_from_file_location("mb4_13_challenge",
                                                  wbm.repo_root() / "solutions" / "challenges" / "mb4_13.py")
    challenge = importlib.util.module_from_spec(spec)   # the hidden part of the challenge: the law of the hidden tables
    spec.loader.exec_module(challenge)                  # and the method of the check (reading it spoils the challenge)
    results, elapsed = [], [0.0]
    report = reporter(results, stop=False)

    def rank_of(label, M):
        start = time.perf_counter()
        got = call_safely(label, hidden_rank_13, M.copy())
        elapsed[0] += time.perf_counter() - start
        if isinstance(got, (bool, np.bool_)) or not isinstance(got, (int, np.integer)) or got < 0:
            print(f"❌ {label} : hidden_rank_13 rend un entier ≥ 0 (un int), reçu {shown(got)}")
            raise StopCheck
        return int(got)

    found = [rank_of(f"Ex MB4.13 : hidden_rank_13 (tableau {k} du notebook)", table)
             for k, table in enumerate(TABLES_13, start=1)]
    report(passes("MB4.13a", found), "les cinq tableaux du notebook : les cinq rangs cachés",
           "les cinq tableaux du notebook : ta fonction ne retrouve pas tous les rangs cachés ; compare chaque valeur "
           "singulière au niveau du bruit, que tu estimes sur le tableau lui-même")
    if not filled(answer_MB4_13a):
        print("⏳ Ex MB4.13a : pas encore fait.")
    else:
        mine = as_floats(answer_MB4_13a)
        report(mine is not None and mine.shape == (5,) and mine.tolist() == found,
               "Ex MB4.13a : ta réponse est ce que rend ta fonction sur les cinq tableaux",
               "Ex MB4.13a : ta réponse n'est pas ce que rend hidden_rank_13 sur les cinq tableaux de TABLES_13 "
               "maintenant (la vérification les recalcule) : exécute à nouveau ta cellule")
    rng = np.random.default_rng()                              # at each run: five new hidden tables, one of each kind
    kinds = list(challenge.KINDS)
    rng.shuffle(kinds)
    for k, kind in enumerate(kinds, start=1):
        seed = int(rng.integers(2 ** 62))
        M = challenge.hidden_table(np.random.default_rng(seed), kind)[0]
        got = rank_of(f"Ex MB4.13 : hidden_rank_13 (tableau caché {k})", M)
        rank = challenge.hidden_table(np.random.default_rng(seed), kind)[1]       # read after the call
        where = f"tableau caché {k} ({kind}, {M.shape[0]} × {M.shape[1]})"
        report(got == rank, f"{where} : le bon rang",
               f"{where} : rendu {got}, alors que le rang caché est {rank} ; un seuil fixe ou une part d'énergie ne suit "
               "pas un bruit dont l'échelle change d'un tableau à l'autre")
    report(elapsed[0] < 10, f"en {fr(elapsed[0], 2)} s pour les dix tableaux",
           f"trop lent : {fr(elapsed[0], 1)} s pour les dix tableaux, il faut moins de 10 s")
    verdict("MB4.13", all(results) and len(results) >= 7, "critère atteint : le bon rang pour les cinq tableaux du "
            "notebook et pour cinq tableaux cachés, en moins de 10 secondes.",
            "critère pas encore atteint : relis les lignes ❌ ci-dessus.")


with checked("MB4.13"):
    check_c13()

**Pour réfléchir** : pourquoi b) et c) ne trouvent-ils pas le rang caché ? Pourquoi une tolérance ou un seuil choisis sur les cinq tableaux du notebook ne suffisent-ils pas pour les tableaux cachés ? Les réponses sont dans `05_solutions.md`, section Notebook.

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu factoriser une matrice de rang $r$ par ses colonnes pivots, en trouver une autre factorisation, et compter les nombres stockés et libres ?
2. Sais-tu mesurer le rang effectif et le rang stable d'un tableau mesuré, et appliquer une matrice factorisée dans l'ordre le moins coûteux ?
3. Sais-tu calculer le poids d'un indice rare, et dire quand le poids corrigé devient négatif ?
4. Sais-tu étudier la saturation et la normalisation par la longueur, et prévoir l'effet de $k_1$ et de $b$ ?
5. Sais-tu fusionner des classements par Borda et par la somme des $\frac{1}{c + \text{rang}}$, et dire pourquoi ces fusions ne dépendent pas des échelles ?
6. Sais-tu calculer recall@k, precision@k et le MRR, et leur valeur pour un ordre tiré au hasard ?

**La suite** : le chapitre bonus B4 de workbookIA, qui applique ces outils.